# Phase 7A — Classical NLP: NLTK & spaCy

**Theory:** Tokenization, stemming, lemmatization, stop words, POS tagging, Named Entity Recognition (NER).

**Install:**
```
pip install nltk spacy
python -m spacy download en_core_web_sm
```

---

In [ ]:
# Check installations
NLTK_AVAILABLE = False
SPACY_AVAILABLE = False

try:
    import nltk

    NLTK_AVAILABLE = True
    print(f"NLTK version: {nltk.__version__}")
except ImportError:
    print("NLTK not installed. Run: pip install nltk")

try:
    import spacy

    SPACY_AVAILABLE = True
    print(f"spaCy version: {spacy.__version__}")
except ImportError:
    print(
        "spaCy not installed. Run: pip install spacy && python -m spacy download en_core_web_sm"
    )

---
## 1. NLTK — The Classical NLP Toolkit

NLTK is the foundation of NLP in Python. Best for learning concepts. For production, use spaCy.

In [ ]:
if NLTK_AVAILABLE:
    # Download required NLTK data
    nltk.download("punkt", quiet=True)
    nltk.download("punkt_tab", quiet=True)
    nltk.download("stopwords", quiet=True)
    nltk.download("wordnet", quiet=True)
    nltk.download("averaged_perceptron_tagger", quiet=True)
    nltk.download("averaged_perceptron_tagger_eng", quiet=True)

    from nltk.tokenize import word_tokenize, sent_tokenize
    from nltk.corpus import stopwords
    from nltk.stem import PorterStemmer, WordNetLemmatizer
    from nltk import pos_tag

    text = "Data scientists use machine learning algorithms to analyze large datasets. They clean, process, and visualize data before building predictive models."

    # 1. Sentence tokenization
    sentences = sent_tokenize(text)
    print("Sentences:")
    for i, s in enumerate(sentences):
        print(f"  [{i}] {s}")

    # 2. Word tokenization
    words = word_tokenize(text)
    print(f"\nWord tokens ({len(words)}): {words[:10]}...")

    # 3. Remove stop words and punctuation
    stop_words = set(stopwords.words("english"))
    filtered = [w.lower() for w in words if w.isalpha() and w.lower() not in stop_words]
    print(f"After removing stop words ({len(filtered)}): {filtered}")

In [ ]:
if NLTK_AVAILABLE:
    # 4. Stemming — cuts words to their root (fast but crude)
    stemmer = PorterStemmer()
    stemmed = [stemmer.stem(w) for w in filtered]
    print("Stemmed:", stemmed[:8])

    # 5. Lemmatization — converts to proper base form (slower but accurate)
    lemmatizer = WordNetLemmatizer()
    lemmatized = [lemmatizer.lemmatize(w, pos="v") for w in filtered]
    print("Lemmatized:", lemmatized[:8])

    # Compare stem vs lemma
    examples = ["running", "flies", "better", "studies", "wolves", "analyses"]
    print("\nWord         Stem          Lemma")
    print("-" * 40)
    for word in examples:
        print(f"{word:12s}   {stemmer.stem(word):12s}   {lemmatizer.lemmatize(word)}")

In [ ]:
if NLTK_AVAILABLE:
    # 6. POS Tagging — assign grammatical role to each word
    tokens = word_tokenize("Machine learning models predict future outcomes.")
    pos_tags = pos_tag(tokens)
    print("POS Tags:")
    for word, tag in pos_tags:
        print(f"  {word:12s} → {tag}")

    print("\nCommon POS tags: NN=noun, VB=verb, JJ=adj, RB=adverb, DT=determiner")

---
## 2. spaCy — Production NLP

In [ ]:
if SPACY_AVAILABLE:
    try:
        nlp = spacy.load("en_core_web_sm")

        text = "Apple CEO Tim Cook announced in San Francisco that the company earned $89 billion in revenue in 2024. The stock rose 3% on Monday."
        doc = nlp(text)

        # Tokens with linguistic annotations
        print("Token Analysis:")
        print(f"{'Token':15s} {'Lemma':15s} {'POS':8s} {'Dep':12s} {'IsStop':8s}")
        print("-" * 60)
        for token in doc[:12]:
            print(
                f"{token.text:15s} {token.lemma_:15s} {token.pos_:8s} {token.dep_:12s} {str(token.is_stop):8s}"
            )

        # Named Entity Recognition
        print("\nNamed Entities:")
        for ent in doc.ents:
            print(f"  '{ent.text}' → {ent.label_} ({spacy.explain(ent.label_)})")

    except OSError:
        print(
            "spaCy model not downloaded. Run: python -m spacy download en_core_web_sm"
        )

---
## 3. Bag of Words Representation

Converting text to numbers is required for ML. The simplest approach: count word occurrences.

In [ ]:
from collections import Counter
import re

# Simple BoW from scratch
corpus = [
    "I love machine learning and deep learning",
    "Machine learning is the future of data science",
    "Deep learning uses neural networks",
    "Data science requires statistics and programming",
]


def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)  # remove punctuation
    stop = {"i", "the", "is", "and", "of", "a", "an", "uses"}
    return [w for w in text.split() if w not in stop]


# Build vocabulary
all_tokens = [t for doc in corpus for t in preprocess(doc)]
vocab = sorted(set(all_tokens))
word2idx = {w: i for i, w in enumerate(vocab)}

# Build BoW matrix
import numpy as np

bow_matrix = np.zeros((len(corpus), len(vocab)), dtype=int)
for i, doc in enumerate(corpus):
    for word in preprocess(doc):
        bow_matrix[i, word2idx[word]] += 1

import pandas as pd

bow_df = pd.DataFrame(bow_matrix, columns=vocab)
print("Bag of Words matrix:")
print(bow_df.to_string())

---
## Summary

| Concept | NLTK | spaCy |
|---------|------|-------|
| Tokenization | `word_tokenize`, `sent_tokenize` | `nlp(text)` → `doc` tokens |
| Stop words | `stopwords.words('english')` | `token.is_stop` |
| Stemming | `PorterStemmer().stem(word)` | Not available (use lemma) |
| Lemmatization | `WordNetLemmatizer().lemmatize(word)` | `token.lemma_` |
| POS tagging | `pos_tag(tokens)` | `token.pos_`, `token.tag_` |
| NER | Requires additional training | `doc.ents`, `ent.label_` |
| Speed | Slower | 10-50x faster |

**When to use:**
- **NLTK**: Learning concepts, research, classic NLP tasks
- **spaCy**: Production systems, fast NER, dependency parsing